# GEC Prompt Engineering Experiment

## Goal
Generate realistic grammatically-erroneous sentences via LLM (Qwen-3 on OpenRouter),
correct them with T5-base-grammar-correction, and evaluate with three complementary metrics.

## Pipeline
```
C4_200M sample (A)  →  LLM generates new erroneous sentences (B)
                    →  T5 corrects B → C
                    →  Eval: CoLA(B) < CoLA(A),  CoLA(C) > CoLA(B),  ERRANT F0.5
```

## Plan steps
1. **Error scope** — 12 ERRANT categories (`gec_error_types.json`)
2. **Zero-shot baseline** — instruction only, no examples
3. **Few-shot stratified** — 3-shot (balanced) and 12-shot (full coverage)
4. **Chain-of-Thought** — explicit error-type identification before generation
5. **Systematic ablation** — one variable at a time → ERRANT F0.5 leaderboard

In [1]:
import os, json, time, re, glob, warnings
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

warnings.filterwarnings('ignore')

# The Jupyter kernel CWD is the workspace root (/workspaces/live-eval),
# not the Denis/ subfolder where .env lives. Try both.
load_dotenv()  # works if kernel CWD == Denis/
if not os.getenv('OPENROUTER_API_KEY'):
    load_dotenv('Denis/.env')  # works if kernel CWD == workspace root

assert os.getenv('OPENROUTER_API_KEY'), 'OPENROUTER_API_KEY not found — check Denis/.env'
print('Imports OK — API key loaded')

/opt/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK — API key loaded


In [2]:
# qwen-3.6 Plus (free tier) via OpenRouter
# Free Qwen models available as of 2026-04: qwen/qwen3-next-80b-a3b-instruct:free
# Fallback if unavailable: 'nvidia/nemotron-3-super-120b-a12b:free'
LLM_MODEL = 'nvidia/nemotron-3-super-120b-a12b:free'

def make_llm(temperature=0.7):
    return ChatOpenAI(
        model=LLM_MODEL,
        api_key=os.getenv('OPENROUTER_API_KEY'),
        base_url='https://openrouter.ai/api/v1',
        temperature=temperature,
    )

llm = make_llm()
print(f'LLM ready: {LLM_MODEL}')

LLM ready: nvidia/nemotron-3-super-120b-a12b:free


In [3]:
with open('gec_error_types.json') as f:
    error_types = json.load(f)['error_types']

with open('gec_few_shot_examples.json') as f:
    few_shot_data = json.load(f)

demonstrations = few_shot_data['demonstrations']   # 12 (input, output) pairs, one per error type
few_shot_examples = few_shot_data['few_shot_examples']  # 3 examples per type for ablation

print(f'Loaded {len(error_types)} error types:')
for et in error_types:
    print(f'  {et["name"]}: "{et["example"]}"')

Loaded 12 error types:
  subject_verb_agreement: "The group of students are going to the library."
  article: "She adopted a old cat from the shelter."
  preposition: "He is good in playing the guitar."
  verb_tense: "Yesterday, she goes to the market and buys vegetables."
  spelling: "The goverment announced new education policies."
  verb_form: "He has went to the store three times this week."
  noun_number: "She gave me a lot of advices about the project."
  pronoun: "Everyone should bring their own lunch, but him forgot."
  word_order: "She always is late to the morning meetings."
  punctuation: "However she decided to stay and finish the report."
  conjunction: "He studied hard, but however he failed the exam."
  adjective: "This is the most cheapest option available in the store."


## Step 1 — Error Scope

All 12 ERRANT categories we target. Using the full set ensures broad benchmark coverage
and avoids bias toward frequent types (SVA, articles dominate C4).

For ablation we also test **balanced-3** (SVA, article, verb_tense) vs **full-12** few-shot coverage
to measure whether more examples always help.

In [4]:
err_df = pd.DataFrame(error_types).rename(columns={'name': 'error_type', 'example': 'example_sentence'})
err_df.index = range(1, len(err_df) + 1)
err_df

,error_type,example_sentence
1,subject_verb_agreement,The group of students are going to the library.
2,article,She adopted a old cat from the shelter.
3,preposition,He is good in playing the guitar.
4,verb_tense,"Yesterday, she goes to the market and buys veg..."
5,spelling,The goverment announced new education policies.
6,verb_form,He has went to the store three times this week.
7,noun_number,She gave me a lot of advices about the project.
8,pronoun,"Everyone should bring their own lunch, but him..."
9,word_order,She always is late to the morning meetings.
10,punctuation,However she decided to stay and finish the rep...


## Step 2 — Data Sampling

`C4_200M` is split across 10 TSV files. To avoid RAM overflow we read each file in
`chunksize=5000` chunks, reservoir-sample `n_rows` per chunk, then downsample to exactly
`n_rows` per file. Peak RAM is bounded by `~chunksize × 2` rows regardless of file size.

| Mode | Per file | Total |
|------|----------|-------|
| Demo | 10 | 100 |
| Full | 100 | 1 000 |

In [5]:
def sample_from_tsv(pattern, n_rows, seed=42, chunksize=5000):
    """Memory-safe sampling from TSV files matching a glob pattern."""
    files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(f'No files matched: {pattern}')

    chunks = []
    for path in tqdm(files, desc='Sampling files'):
        reservoir = []
        for chunk in pd.read_csv(
            path, sep='\t', header=None,
            names=['original', 'ground_truth'],
            chunksize=chunksize
        ):
            n = min(n_rows, len(chunk))
            reservoir.append(chunk.sample(n=n, random_state=seed))
        file_sample = pd.concat(reservoir).sample(n=n_rows, random_state=seed).reset_index(drop=True)
        chunks.append(file_sample)
        del reservoir, file_sample

    return pd.concat(chunks, ignore_index=True)

DEMO_MODE = True   # flip to False for full 1000-sentence run
N_PER_FILE = 10 if DEMO_MODE else 100
print(f'Mode: {"demo" if DEMO_MODE else "full"} — {N_PER_FILE} per file × 10 files = {N_PER_FILE * 10} total')

Mode: demo — 10 per file × 10 files = 100 total


In [6]:
df = sample_from_tsv('data/C4_200M/C4_200M.tsv-*', n_rows=N_PER_FILE)
src = df['original'].tolist()
gt  = df['ground_truth'].tolist()
print(f'Sampled {len(df)} rows total')
df.head(3)

Sampling files:   0%|          | 0/10 [00:00<?, ?it/s]

Sampling files:  10%|█         | 1/10 [01:51<16:41, 111.31s/it]

Sampling files:  20%|██        | 2/10 [03:30<13:51, 103.94s/it]

Sampling files:  30%|███       | 3/10 [05:08<11:49, 101.29s/it]

Sampling files:  40%|████      | 4/10 [06:47<10:01, 100.32s/it]

...

Sampling files:  90%|█████████ | 9/10 [15:21<01:43, 103.75s/it]

Sampling files: 100%|██████████| 10/10 [17:09<00:00, 105.07s/it]

Sampling files: 100%|██████████| 10/10 [17:09<00:00, 102.99s/it]

Sampled 100 rows total


,original,ground_truth
0,Vicinity and Keppel will they hold a 50 per ce...,Vicinity and Keppel will each hold a 50 per ce...
1,"Most smart contracts, it turns out, are not pe...","Most smart contracts, it turns out, are not pe..."
2,Is this the smallest conrtoller! door you’ve e...,Is this the smallest door controller you’ve ev...


## Shared Batch Generation Utilities

All prompt variants share the same batched generation loop.
The model receives `BATCH_SIZE` numbered sentences and returns one generated sentence per line.
Retry logic handles rate limits (exponential back-off) and output parse failures.

In [7]:
from openai import RateLimitError

BATCH_SIZE = 5
SLEEP_SEC  = 10

def make_batch_input(sentences):
    return '\n'.join(f'{i+1}. {s}' for i, s in enumerate(sentences))

def parse_batch_output(text, expected):
    """Extract one output line per input sentence from numbered LLM output."""
    lines = [
        re.sub(r'^[0-9]+[.)\s]+', '', l).strip()
        for l in text.strip().splitlines()
        if re.match(r'^[0-9]+[.)]', l.strip())
    ]
    if len(lines) == expected:
        return lines
    # fallback: model returned plain unnumbered lines
    fallback = [l.strip() for l in text.strip().splitlines() if l.strip()]
    if len(fallback) == expected:
        return fallback
    raise ValueError(f'Expected {expected} lines, got {len(lines)} numbered / {len(fallback)} plain:\n{text}')

def invoke_batch_with_retry(chain, sentences, max_retries=7, base_delay=10):
    batch_input = make_batch_input(sentences)
    last_error = None
    for attempt in range(max_retries):
        try:
            result = chain.invoke({'batch_input': batch_input})
            return parse_batch_output(result, len(sentences))
        except ValueError as e:
            last_error = e
            time.sleep(base_delay * (2 ** attempt))
        except RateLimitError as e:
            wait = 60 * (2 ** attempt)
            print(f'\nRate limit, retrying in {wait}s...')
            time.sleep(wait)
    raise RuntimeError(f'Failed after {max_retries} retries. Last: {last_error}')

def run_generation(chain, sentences, batch_size=BATCH_SIZE, sleep=SLEEP_SEC, desc='Generating'):
    batches = [sentences[i:i+batch_size] for i in range(0, len(sentences), batch_size)]
    results = []
    for batch in tqdm(batches, desc=desc):
        results.extend(invoke_batch_with_retry(chain, batch))
        time.sleep(sleep)
    return results

print('Batch utilities ready')

Batch utilities ready


## Prompt Variants

### 2.1 Zero-Shot Baseline

Instruction only — no examples. The model must infer the task from text alone.
Sets the **lower bound** for ablations.

**Expected weakness**: without examples the model may correct errors rather than replicate them,
or generate generic incorrect sentences that don't match C4 error distributions.

In [8]:
ZS_INSTR = (
    'You are a grammatical error generator. '
    'For each sentence below, write a completely NEW sentence on a different topic '
    'that contains the same type of grammatical error. '
    'Do NOT correct the given sentence — create an entirely new one. '
    'Return only the generated sentences, one per line, preserving the original numbering.'
)

zero_shot_chain = (
    ChatPromptTemplate.from_messages([('human', ZS_INSTR + '\n\n{batch_input}')])
    | llm
    | StrOutputParser()
)
print('Zero-shot chain ready')

Zero-shot chain ready


### 2.2 One-Shot

One SVA example shows the model the expected **input/output format**
(erroneous in → different erroneous out on a new topic) without revealing all error categories.

**Design choice**: single example from `gec_error_types.json` keeps it taxonomy-consistent
and cheap (short prompt). Main purpose is clarifying the format, not the error space.

In [9]:
ONE_SHOT_EX = (
    'Example:\n'
    'Input:  1. The group of students are going to the library.\n'
    'Output: 1. The team of engineers was arguing about their blueprints.\n\n'
)

one_shot_chain = (
    ChatPromptTemplate.from_messages([('human', ZS_INSTR + '\n\n' + ONE_SHOT_EX + '{batch_input}')])
    | llm
    | StrOutputParser()
)
print('One-shot chain ready')

One-shot chain ready


### 3 Few-Shot (Stratified by Error Type)

Two coverage levels from `gec_few_shot_examples.json → demonstrations`:

- **3-shot balanced**: SVA, article, verb_tense — three most frequent ERRANT categories in C4.
- **12-shot full**: one `(input, output)` demonstration per error type.

**Hypothesis**: more examples reduce hallucination (model correcting instead of replicating)
but add prompt length overhead → longer batch processing, smaller batch sizes needed.
Ablation will test whether 3→12 examples is worth the extra tokens.

In [10]:
def make_few_shot_block(pairs):
    """Format list of (input, output) pairs as a numbered examples block."""
    lines = []
    for i, (inp, out) in enumerate(pairs, 1):
        lines += [f'Example {i}:', f'Input:  {inp}', f'Output: {out}', '']
    return '\n'.join(lines)

FS_BASE_INSTR = (
    'You are a grammatical error generator. '
    'For each sentence below, write a completely NEW sentence on a different topic '
    'that contains the same type of grammatical error as the input. '
    'Do NOT correct the given sentence.\n'
    'Return only the generated sentences, one per line, preserving the numbering.\n\n'
)

def make_few_shot_chain(pairs, temperature=0.7):
    block = make_few_shot_block(pairs)
    prompt = ChatPromptTemplate.from_messages([
        ('human', FS_BASE_INSTR + block + 'Now generate for:\n{batch_input}')
    ])
    return prompt | make_llm(temperature) | StrOutputParser()

# Balanced 3-shot: SVA, article, verb_tense
pairs_3 = [
    (demonstrations[k]['input'], demonstrations[k]['output'])
    for k in ('subject_verb_agreement', 'article', 'verb_tense')
]

# Full 12-shot: one per error type (same order as gec_error_types.json)
pairs_12 = [
    (demonstrations[et['name']]['input'], demonstrations[et['name']]['output'])
    for et in error_types
]

few_shot_3_chain  = make_few_shot_chain(pairs_3)
few_shot_12_chain = make_few_shot_chain(pairs_12)
print(f'Few-shot chains ready:  3-shot ({len(pairs_3)} examples),  12-shot ({len(pairs_12)} examples)')

Few-shot chains ready:  3-shot (3 examples),  12-shot (12 examples)

### 4 Chain-of-Thought (CoT)

The model is asked to:
1. Name the grammatical error type in the input sentence.
2. Generate a new sentence with the same error type.

**Rationale**: naming the type acts as a soft constraint — it anchors the generation
to the right error category before producing the surface form.

**Expected benefit**: higher precision on specific error categories, fewer hallucinated corrections.

**Expected cost**: structured two-part output is harder to parse;
batch size must be smaller (3) to reduce parse failures.

Output format required: `[N] Error type: <type>` then `[N] Generated: <sentence>`.

In [11]:
COT_INSTR = (
    'You are a grammatical error generator. For each numbered sentence below:\n'
    '  Step 1: Identify the grammatical error type (e.g. subject-verb agreement, article, spelling).\n'
    '  Step 2: Write a completely NEW sentence on a different topic with the same error type.\n\n'
    'Use this exact format for each sentence:\n'
    '[N] Error type: <type>\n'
    '[N] Generated: <new sentence>\n\n'
    'Do NOT correct or rewrite the input sentence.\n'
)

cot_chain = (
    ChatPromptTemplate.from_messages([('human', COT_INSTR + '\n{batch_input}')])
    | make_llm(0.7)
    | StrOutputParser()
)

def parse_cot_output(text, expected):
    """Extract Generated: lines from CoT structured output."""
    generated = []
    for line in text.strip().splitlines():
        m = re.match(r'^\[[0-9]+\]\s*Generated:\s*(.+)', line.strip())
        if m:
            generated.append(m.group(1).strip())
    if len(generated) == expected:
        return generated
    # Fallback: any line containing 'Generated:'
    generated = [
        line.split('Generated:', 1)[1].strip()
        for line in text.strip().splitlines()
        if 'Generated:' in line
    ]
    if len(generated) == expected:
        return generated
    raise ValueError(f'CoT parse failed. Expected {expected}, got {len(generated)}:\n{text}')

def run_cot(sentences, batch_size=3, sleep=20, desc='CoT'):
    batches = [sentences[i:i+batch_size] for i in range(0, len(sentences), batch_size)]
    results = []
    failed_batches = []
    for batch_idx, batch in enumerate(tqdm(batches, desc=desc)):
        batch_input = make_batch_input(batch)
        success = False
        for attempt in range(7):
            try:
                out = cot_chain.invoke({'batch_input': batch_input})
                results.extend(parse_cot_output(out, len(batch)))
                success = True
                break
            except ValueError:
                time.sleep(10 * (2 ** attempt))
            except RateLimitError:
                time.sleep(60 * (2 ** attempt))
        if not success:
            # Keep length aligned so downstream cells never crash on mismatch.
            # Placeholders are easy to filter out before evaluation.
            results.extend(['[GENERATION FAILED]'] * len(batch))
            failed_batches.append(batch_idx)
        time.sleep(sleep)
    if failed_batches:
        print(f'  Warning: {len(failed_batches)} batch(es) used placeholders: idx {failed_batches}')
    return results

print('CoT chain ready')

CoT chain ready


## Generation Pipeline

Run all 5 variants on the same sample and save to individual CSVs.
CSVs are the checkpoint: if the kernel dies mid-run, re-load from CSV and skip re-generation.

> Tune `BATCH_SIZE` and `SLEEP_SEC` above if you hit rate limits on the free tier.
> For 12-shot and CoT, use `batch_size=3` to stay within token limits.

## Generation Pipeline

Each variant runs in its own cell — interrupt and resume individually without losing completed work.
Results checkpoint to CSV after each variant.

In [12]:
src = df['original'].tolist()
gt  = df['ground_truth'].tolist()
results = {}
print(f'{len(src)} sentences loaded')

100 sentences loaded


In [13]:
print('--- Zero-shot ---')
results['zero_shot'] = run_generation(zero_shot_chain, src, desc='Zero-shot')
pd.DataFrame({'original': src, 'generated': results['zero_shot'], 'ground_truth': gt}).to_csv('gen_zero_shot.csv', index=False)
print(f'Saved gen_zero_shot.csv  ({len(results["zero_shot"])} rows)')

--- Zero-shot ---


Zero-shot:   0%|          | 0/20 [00:00<?, ?it/s]

Zero-shot:   5%|▌         | 1/20 [01:03<19:59, 63.12s/it]

Zero-shot:  10%|█         | 2/20 [01:43<14:56, 49.82s/it]

Zero-shot:  15%|█▌        | 3/20 [03:11<19:02, 67.19s/it]

...

Zero-shot:  90%|█████████ | 18/20 [17:43<01:56, 58.25s/it]

Zero-shot:  95%|█████████▌| 19/20 [18:35<00:56, 56.31s/it]

Zero-shot: 100%|██████████| 20/20 [19:11<00:00, 50.08s/it]

Zero-shot: 100%|██████████| 20/20 [19:11<00:00, 57.56s/it]

Saved gen_zero_shot.csv  (100 rows)


In [14]:
print('--- One-shot ---')
results['one_shot'] = run_generation(one_shot_chain, src, desc='One-shot')
pd.DataFrame({'original': src, 'generated': results['one_shot'], 'ground_truth': gt}).to_csv('gen_one_shot.csv', index=False)
print(f'Saved gen_one_shot.csv  ({len(results["one_shot"])} rows)')

--- One-shot ---


One-shot:   0%|          | 0/20 [00:00<?, ?it/s]

One-shot:   5%|▌         | 1/20 [01:06<21:07, 66.71s/it]

One-shot:  10%|█         | 2/20 [02:24<22:02, 73.46s/it]

One-shot:  15%|█▌        | 3/20 [03:47<21:57, 77.51s/it]

...

One-shot:  90%|█████████ | 18/20 [20:32<02:14, 67.38s/it]

One-shot:  95%|█████████▌| 19/20 [21:59<01:13, 73.30s/it]

One-shot: 100%|██████████| 20/20 [23:14<00:00, 73.73s/it]

One-shot: 100%|██████████| 20/20 [23:14<00:00, 69.73s/it]

Saved gen_one_shot.csv  (100 rows)


In [15]:
print('--- Few-shot 3 ---')
results['few_shot_3'] = run_generation(few_shot_3_chain, src, batch_size=3, sleep=15, desc='Few-shot-3')
pd.DataFrame({'original': src, 'generated': results['few_shot_3'], 'ground_truth': gt}).to_csv('gen_few_shot_3.csv', index=False)
print(f'Saved gen_few_shot_3.csv  ({len(results["few_shot_3"])} rows)')

--- Few-shot 3 ---


Few-shot-3:   0%|          | 0/34 [00:00<?, ?it/s]

Few-shot-3:   3%|▎         | 1/34 [01:35<52:16, 95.05s/it]

Few-shot-3:   6%|▌         | 2/34 [02:38<40:52, 76.63s/it]

Few-shot-3:   9%|▉         | 3/34 [03:31<33:55, 65.65s/it]

...

Few-shot-3:  94%|█████████▍| 32/34 [38:00<01:50, 55.24s/it]

Few-shot-3:  97%|█████████▋| 33/34 [39:34<01:06, 66.74s/it]

Few-shot-3: 100%|██████████| 34/34 [40:16<00:00, 59.41s/it]

Few-shot-3: 100%|██████████| 34/34 [40:16<00:00, 71.07s/it]

Saved gen_few_shot_3.csv  (100 rows)


In [16]:
print('--- Few-shot 12 ---')
results['few_shot_12'] = run_generation(few_shot_12_chain, src, batch_size=3, sleep=20, desc='Few-shot-12')
pd.DataFrame({'original': src, 'generated': results['few_shot_12'], 'ground_truth': gt}).to_csv('gen_few_shot_12.csv', index=False)
print(f'Saved gen_few_shot_12.csv  ({len(results["few_shot_12"])} rows)')

--- Few-shot 12 ---


Few-shot-12:   0%|          | 0/34 [00:00<?, ?it/s]

Few-shot-12:   3%|▎         | 1/34 [01:29<49:05, 89.25s/it]

Few-shot-12:   6%|▌         | 2/34 [02:12<33:12, 62.26s/it]

Few-shot-12:   9%|▉         | 3/34 [03:20<33:34, 65.00s/it]

...

Few-shot-12:  94%|█████████▍| 32/34 [47:35<02:42, 81.21s/it]

Few-shot-12:  97%|█████████▋| 33/34 [48:50<01:19, 79.40s/it]

Few-shot-12: 100%|██████████| 34/34 [49:14<00:00, 62.68s/it]

Few-shot-12: 100%|██████████| 34/34 [49:14<00:00, 86.88s/it]

Saved gen_few_shot_12.csv  (100 rows)


In [17]:
print('--- CoT ---')
results['cot'] = run_cot(src)
pd.DataFrame({'original': src, 'generated': results['cot'], 'ground_truth': gt}).to_csv('gen_cot.csv', index=False)
print(f'Saved gen_cot.csv  ({len(results["cot"])} rows)')

--- CoT ---


CoT:   0%|          | 0/34 [00:00<?, ?it/s]

CoT:   3%|▎         | 1/34 [00:47<26:14, 47.72s/it]

CoT:   6%|▌         | 2/34 [01:25<22:12, 41.64s/it]

CoT:   9%|▉         | 3/34 [02:22<25:08, 48.67s/it]

...

CoT:  94%|█████████▍| 32/34 [28:23<01:36, 48.28s/it]

CoT:  97%|█████████▋| 33/34 [30:06<01:04, 64.52s/it]

CoT: 100%|██████████| 34/34 [30:41<00:00, 55.72s/it]

CoT: 100%|██████████| 34/34 [30:41<00:00, 54.16s/it]

Saved gen_cot.csv  (100 rows)


In [18]:
# Reload any variants from CSV in case kernel was restarted mid-run
VARIANTS = ['zero_shot', 'one_shot', 'few_shot_3', 'few_shot_12', 'cot']
for v in VARIANTS:
    if v not in results:
        import os
        if os.path.exists(f'gen_{v}.csv'):
            df_v = pd.read_csv(f'gen_{v}.csv')
            results[v] = df_v['generated'].tolist()
            print(f'Reloaded {v} from CSV ({len(results[v])} rows)')
        else:
            print(f'Missing: {v} — run the cell above')
print('results keys:', list(results.keys()))

results keys: ['zero_shot', 'one_shot', 'few_shot_3', 'few_shot_12', 'cot']


## T5 GEC Correction  (B → C)

Use `vennify/t5-base-grammar-correction` (seq2seq T5) to correct LLM-generated sentences.

**Why seq2seq directly, not `pipeline('text-generation')`**:  
transformers 5.x `text-generation` pipeline only supports causal LMs;
T5 requires `AutoModelForSeq2SeqLM` with `.generate()`.  
The input prefix `"grammar: "` is required by the model's training format.

In [19]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

T5_MODEL = 'vennify/t5-base-grammar-correction'
t5_tok = AutoTokenizer.from_pretrained(T5_MODEL)
t5_mdl = AutoModelForSeq2SeqLM.from_pretrained(T5_MODEL)
t5_mdl.eval()

def correct_sentences(sentences, batch_size=8, prefix='grammar: '):
    results = []
    for i in tqdm(range(0, len(sentences), batch_size), desc='T5 GEC'):
        batch = [prefix + s for s in sentences[i:i+batch_size]]
        inputs = t5_tok(batch, return_tensors='pt', padding=True, truncation=True, max_length=128)
        with torch.no_grad():
            outputs = t5_mdl.generate(**inputs, max_new_tokens=128)
        results.extend(t5_tok.batch_decode(outputs, skip_special_tokens=True))
    return results

print('T5 model loaded')

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 260/260 [00:00<00:00, 11875.02it/s]


The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


T5 model loaded


In [20]:
corrected = {}
for variant in results:
    print(f'\nRunning T5 on: {variant}')
    df_v = pd.read_csv(f'gen_{variant}.csv')
    df_v['t5_corrected'] = correct_sentences(df_v['generated'].tolist())
    df_v.to_csv(f'gen_{variant}_corrected.csv', index=False)
    corrected[variant] = df_v
    print(f'  orig:  {df_v["generated"].iloc[0]}')
    print(f'  fixed: {df_v["t5_corrected"].iloc[0]}')


Running T5 on: zero_shot


T5 GEC:   0%|          | 0/13 [00:00<?, ?it/s]

T5 GEC:   8%|▊         | 1/13 [00:06<01:19,  6.60s/it]

T5 GEC:  15%|█▌        | 2/13 [00:09<00:48,  4.45s/it]

T5 GEC:  23%|██▎       | 3/13 [00:17<00:58,  5.87s/it]

...

T5 GEC:  85%|████████▍ | 11/13 [00:16<00:03,  1.51s/it]

T5 GEC:  92%|█████████▏| 12/13 [00:17<00:01,  1.49s/it]

T5 GEC: 100%|██████████| 13/13 [00:18<00:00,  1.27s/it]

T5 GEC: 100%|██████████| 13/13 [00:18<00:00,  1.42s/it]

  orig:  The group of students are going to the museum tomorrow.
  fixed: The group of students are going to the museum tomorrow.


## Evaluation

Three complementary metrics:

| Metric | What it measures | Implementation |
|--------|-----------------|----------------|
| CoLA | Grammatical acceptability of A, B, C | `textattack/bert-base-uncased-CoLA` |
| Semantic similarity | Topic preservation: original ↔ generated | TF-IDF cosine (sklearn, no extra deps) |
| ERRANT F0.5 | Error-type distribution alignment vs C4 reference | `errant` package |

**Key assertions** (per variant):  
- `CoLA(B) < CoLA(A)` — generated sentences are erroneous (LLM task success)  
- `CoLA(C) > CoLA(B)` — T5 genuinely improves B (T5 task success)

### CoLA Scores

- **A** = C4 original (already erroneous — erroneous baseline)
- **B** = LLM-generated (should be erroneous: `CoLA(B) ≤ CoLA(A)`)
- **C** = T5-corrected B (should improve: `CoLA(C) > CoLA(B)`)

CoLA outputs binary acceptability (1 = acceptable, 0 = unacceptable).
We compute the mean over each set as a grammaticality rate.

In [21]:
from transformers import pipeline as hf_pipeline

device = 0 if torch.cuda.is_available() else -1
cola_scorer = hf_pipeline(
    'text-classification',
    model='textattack/bert-base-uncased-CoLA',
    device=device,
)

def cola_scores(texts, batch_size=32):
    """Return list of 1.0 (acceptable) or 0.0 (unacceptable) per sentence."""
    scores = []
    for i in tqdm(range(0, len(texts), batch_size), desc='CoLA'):
        preds = cola_scorer(texts[i:i+batch_size], truncation=True, max_length=128)
        scores.extend(1.0 if p['label'] == 'acceptable' else 0.0 for p in preds)
    return scores

cola_results = {}
for variant, df_v in corrected.items():
    a = cola_scores(df_v['original'].tolist())
    b = cola_scores(df_v['generated'].tolist())
    c = cola_scores(df_v['t5_corrected'].tolist())
    cola_results[variant] = {
        'cola_A': round(np.mean(a), 3),
        'cola_B': round(np.mean(b), 3),
        'cola_C': round(np.mean(c), 3),
        'B<A': bool(np.mean(b) < np.mean(a)),
        'C>B': bool(np.mean(c) > np.mean(b)),
    }

cola_df = pd.DataFrame(cola_results).T
print('CoLA results (higher = more grammatically acceptable):')
print(cola_df.to_string())

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 16212.91it/s]

CoLA:   0%|          | 0/4 [00:00<?, ?it/s]

CoLA:  25%|██▌       | 1/4 [00:03<00:09,  3.17s/it]

CoLA:  50%|█████     | 2/4 [00:05<00:05,  2.57s/it]

...

CoLA:  50%|█████     | 2/4 [00:02<00:02,  1.40s/it]

CoLA:  75%|███████▌  | 3/4 [00:04<00:01,  1.37s/it]

CoLA: 100%|██████████| 4/4 [00:04<00:00,  1.12it/s]

CoLA: 100%|██████████| 4/4 [00:04<00:00,  1.08s/it]

CoLA results (higher = more grammatically acceptable):
            cola_A cola_B cola_C    B<A    C>B
zero_shot      0.0    0.0    0.0  False  False
one_shot       0.0    0.0    0.0  False  False
few_shot_3     0.0    0.0    0.0  False  False
few_shot_12    0.0    0.0    0.0  False  False
cot            0.0    0.0    0.0  False  False


### Semantic Similarity

TF-IDF cosine similarity between the original C4 sentence (A) and the LLM-generated sentence (B).

**Implementation choice**: sklearn TF-IDF avoids adding `sentence-transformers` as a dependency
(not in `requirements.txt`). It's a lightweight proxy — captures vocabulary overlap,
not deep semantics, but sufficient to flag extreme cases.

**Target range**: 0.15 – 0.50. Too high (> 0.8) = paraphrase; too low (< 0.1) = off-topic.

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

def tfidf_sim(texts_a, texts_b):
    """Pairwise TF-IDF cosine similarity between two aligned lists of texts."""
    all_texts = texts_a + texts_b
    tfidf = TfidfVectorizer().fit_transform(all_texts)
    n = len(texts_a)
    return [float(cos_sim(tfidf[i], tfidf[n + i])[0, 0]) for i in range(n)]

sem_results = {}
for variant, df_v in corrected.items():
    sims = tfidf_sim(df_v['original'].tolist(), df_v['generated'].tolist())
    sem_results[variant] = {
        'mean_sim': round(np.mean(sims), 3),
        'std_sim':  round(np.std(sims),  3),
    }

sem_df = pd.DataFrame(sem_results).T
print('Semantic similarity  original → generated  (TF-IDF cosine):')
print(sem_df.to_string())

Semantic similarity  original → generated  (TF-IDF cosine):
             mean_sim  std_sim
zero_shot       0.112    0.121
one_shot        0.094    0.106
few_shot_3      0.089    0.135
few_shot_12     0.054    0.068
cot             0.033    0.041


### ERRANT — Error-Type Distribution

ERRANT annotates grammatical edits and classifies them into ERRANT types (R:SVA, U:ART, M:PREP, etc.).

**What we compute**:
1. **Reference distribution** — edit types in C4 original (A) → ground_truth corrections
2. **Hypothesis distribution** — edit types in LLM-generated (B) → T5-corrected (C)
3. **F0.5 on normalized type distributions** — precision-weighted overlap

**Why F0.5?** Standard GEC metric; weights precision 2× recall.
Here: generating the *right error types* (precision) matters more than generating *all* types (recall).

**Interpretation**: high F0.5 = LLM generates the same kinds of errors as the real C4 dataset.

In [23]:
import spacy, errant

# Download spacy model if missing
try:
    spacy.load('en_core_web_sm')
except OSError:
    import subprocess
    subprocess.run(['python', '-m', 'spacy', 'download', 'en_core_web_sm'], check=True)

annotator = errant.load('en')

def get_edits(src_sent, tgt_sent):
    orig = annotator.parse(src_sent)
    cor  = annotator.parse(tgt_sent)
    return annotator.annotate(orig, cor)

def edit_type_dist(edit_lists):
    """Normalized error-type distribution over a list of edit lists."""
    from collections import Counter
    counter = Counter(e.type for edits in edit_lists for e in edits)
    total = sum(counter.values()) or 1
    return {k: v / total for k, v in counter.items()}

def dist_f05(hyp_dist, ref_dist):
    """F0.5 on normalized error-type distributions."""
    types = set(hyp_dist) | set(ref_dist)
    tp = sum(min(hyp_dist.get(t, 0), ref_dist.get(t, 0)) for t in types)
    fp = sum(max(hyp_dist.get(t, 0) - ref_dist.get(t, 0), 0) for t in types)
    fn = sum(max(ref_dist.get(t, 0) - hyp_dist.get(t, 0), 0) for t in types)
    p  = tp / (tp + fp) if tp + fp > 0 else 0
    r  = tp / (tp + fn) if tp + fn > 0 else 0
    return (1.25 * p * r) / (0.25 * p + r) if 0.25 * p + r > 0 else 0

# Reference distribution: C4 original → C4 ground_truth
first_df = corrected[list(corrected)[0]]
ref_edits = [
    get_edits(o, g)
    for o, g in tqdm(
        zip(first_df['original'].tolist(), first_df['ground_truth'].tolist()),
        desc='Ref ERRANT', total=len(first_df)
    )
]
ref_dist = edit_type_dist(ref_edits)

print('Reference error-type distribution (top 10):')
for k, v in sorted(ref_dist.items(), key=lambda x: -x[1])[:10]:
    print(f'  {k}: {v:.3f}')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/12.8 MB ? eta -:--:--

     ━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/12.8 MB 7.1 MB/s eta 0:00:02

     ━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/12.8 MB 8.0 MB/s eta 0:00:02

     ━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/12.8 MB 7.2 MB/s eta 0:00:02

     ━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━ 5.2/12.8 MB 6.5 MB/s eta 0:00:02

...

Ref ERRANT:  79%|███████▉  | 79/100 [00:01<00:00, 70.60it/s]

Ref ERRANT:  87%|████████▋ | 87/100 [00:01<00:00, 71.67it/s]

Ref ERRANT:  95%|█████████▌| 95/100 [00:01<00:00, 60.64it/s]

Ref ERRANT: 100%|██████████| 100/100 [00:01<00:00, 66.16it/s]

Reference error-type distribution (top 10):
  R:OTHER: 0.199
  R:NOUN: 0.143
  R:SPELL: 0.060
  M:DET: 0.058
  R:ORTH: 0.048
  R:PREP: 0.046
  R:VERB: 0.042
  R:VERB:FORM: 0.035
  R:NOUN:NUM: 0.028
  R:VERB:TENSE: 0.028


In [24]:
errant_results = {}
for variant, df_v in corrected.items():
    hyp_edits = [
        get_edits(g, t)
        for g, t in tqdm(
            zip(df_v['generated'].tolist(), df_v['t5_corrected'].tolist()),
            desc=f'ERRANT {variant}', total=len(df_v)
        )
    ]
    hyp_dist = edit_type_dist(hyp_edits)
    errant_results[variant] = {
        'f0.5':    round(dist_f05(hyp_dist, ref_dist), 3),
        'n_edits': round(np.mean([len(e) for e in hyp_edits]), 2),
    }

# Combined leaderboard
leaderboard = pd.DataFrame({
    v: {
        'f0.5':    errant_results[v]['f0.5'],
        'n_edits': errant_results[v]['n_edits'],
        'cola_B':  cola_results[v]['cola_B'],
        'cola_C':  cola_results[v]['cola_C'],
        'sem_sim': sem_results[v]['mean_sim'],
        'B<A':     cola_results[v]['B<A'],
        'C>B':     cola_results[v]['C>B'],
    }
    for v in errant_results
}).T.sort_values('f0.5', ascending=False)

print('\n=== PROMPT LEADERBOARD (sorted by ERRANT F0.5) ===')
print(leaderboard.to_string())
leaderboard.to_csv('prompt_leaderboard.csv')
print('\nSaved prompt_leaderboard.csv')

ERRANT zero_shot:   0%|          | 0/100 [00:00<?, ?it/s]

ERRANT zero_shot:  12%|█▏        | 12/100 [00:00<00:00, 115.08it/s]

ERRANT zero_shot:  24%|██▍       | 24/100 [00:00<00:00, 97.65it/s] 

ERRANT zero_shot:  36%|███▌      | 36/100 [00:00<00:00, 105.62it/s]

ERRANT zero_shot:  51%|█████     | 51/100 [00:00<00:00, 120.99it/s]

...

ERRANT cot:  68%|██████▊   | 68/100 [00:00<00:00, 158.71it/s]

ERRANT cot:  84%|████████▍ | 84/100 [00:00<00:00, 156.56it/s]

ERRANT cot: 100%|██████████| 100/100 [00:00<00:00, 155.26it/s]

ERRANT cot: 100%|██████████| 100/100 [00:00<00:00, 157.53it/s]


=== PROMPT LEADERBOARD (sorted by ERRANT F0.5) ===
              f0.5 n_edits cola_B cola_C sem_sim    B<A    C>B
zero_shot    0.689    1.19    0.0    0.0   0.112  False  False
one_shot     0.659    0.97    0.0    0.0   0.094  False  False
few_shot_3   0.639    0.94    0.0    0.0   0.089  False  False
few_shot_12  0.591    0.78    0.0    0.0   0.054  False  False
cot          0.468    0.69    0.0    0.0   0.033  False  False

Saved prompt_leaderboard.csv


## Step 5 — Ablation Study

Change **one variable at a time** vs the `few_shot_12` baseline.
Run on `N_ABL = min(30, len(src))` sentences to conserve API quota.

| Variable | Values tested | Fixed |
|----------|--------------|-------|
| Instruction wording | concise vs detailed | T=0.7, 12-shot |
| Temperature | 0.3 / 0.7 / 1.0 | detailed instr, 12-shot |
| n_examples | 0 / 1 / 3 / 12 | detailed instr, T=0.7 |
| CoT | off / on | detailed instr, T=0.7 |

n_examples 0/1/3/12 and CoT results come from the main run above (sliced to N_ABL).

In [25]:
N_ABL = min(30, len(src))
abl_src = src[:N_ABL]
abl_gt  = gt[:N_ABL]

# ── Instruction wording ──────────────────────────────────────────────────────
INSTR_CONCISE = (
    'Generate a new sentence with the same grammatical error type. '
    'New sentence only, numbered.'
)
INSTR_DETAILED = (
    'You are an expert linguist specializing in grammatical error generation. '
    'Create a completely new, original sentence that exhibits the same category of '
    'grammatical error as the given sentence. Use an entirely different topic. '
    'Preserve the original numbering. Output only the generated sentences.'
)

NEW_ABLATION_CONFIGS = {
    # Instruction wording (12-shot, T=0.7)
    'instr_concise':  make_few_shot_chain(pairs_12, temperature=0.7),  # swap instruction below
    'instr_detailed': make_few_shot_chain(pairs_12, temperature=0.7),
    # Temperature (12-shot, detailed instr, T=0.3 / 1.0 — 0.7 covered by few_shot_12 main)
    'temp_0.3': make_few_shot_chain(pairs_12, temperature=0.3),
    'temp_1.0': make_few_shot_chain(pairs_12, temperature=1.0),
}

# Patch instruction wording into concise / detailed chains
def make_wording_chain(instr, pairs, temperature=0.7):
    block = make_few_shot_block(pairs)
    prompt = ChatPromptTemplate.from_messages([
        ('human', instr + '\n\n' + block + 'Now generate for:\n{batch_input}')
    ])
    return prompt | make_llm(temperature) | StrOutputParser()

NEW_ABLATION_CONFIGS['instr_concise']  = make_wording_chain(INSTR_CONCISE,  pairs_12)
NEW_ABLATION_CONFIGS['instr_detailed'] = make_wording_chain(INSTR_DETAILED, pairs_12)

print(f'Ablation: {len(NEW_ABLATION_CONFIGS)} new configs + main run slices')
print(f'Sentences per config: {N_ABL}')

Ablation: 4 new configs + main run slices
Sentences per config: 30


In [26]:
abl_rows = []

def eval_generated(name, generated, orig_list, gt_list):
    """Score a generated list with T5, CoLA, semantic sim, ERRANT and return a metrics dict."""
    t5_c = correct_sentences(generated)
    a_sc = cola_scores(orig_list)
    b_sc = cola_scores(generated)
    c_sc = cola_scores(t5_c)
    sims = tfidf_sim(orig_list, generated)
    h_edits = [get_edits(g, t) for g, t in zip(generated, t5_c)]
    r_edits = [get_edits(o, g) for o, g in zip(orig_list, gt_list)]
    h_dist  = edit_type_dist(h_edits)
    r_dist  = edit_type_dist(r_edits)
    return {
        'config':  name,
        'f0.5':    round(dist_f05(h_dist, r_dist), 3),
        'cola_B':  round(np.mean(b_sc), 3),
        'cola_C':  round(np.mean(c_sc), 3),
        'sem_sim': round(np.mean(sims),  3),
        'B<A':     bool(np.mean(b_sc) < np.mean(a_sc)),
        'C>B':     bool(np.mean(c_sc) > np.mean(b_sc)),
    }

# Include main-run variants (sliced to N_ABL) for comparison
MAIN_SLICE_VARIANTS = ['zero_shot', 'one_shot', 'few_shot_3', 'few_shot_12', 'cot']
for v in MAIN_SLICE_VARIANTS:
    if v in results:
        abl_rows.append(eval_generated(v, results[v][:N_ABL], abl_src, abl_gt))
        print(f'Evaluated (main slice): {v}')

# Run new ablation configs
for name, chain in NEW_ABLATION_CONFIGS.items():
    print(f'\n--- {name} ---')
    try:
        generated = run_generation(chain, abl_src, batch_size=3, sleep=15, desc=name)
        abl_rows.append(eval_generated(name, generated, abl_src, abl_gt))
        print(f'  f0.5 = {abl_rows[-1]["f0.5"]}')
    except Exception as e:
        print(f'  FAILED: {e}')

abl_df = pd.DataFrame(abl_rows).set_index('config').sort_values('f0.5', ascending=False)
print('\n=== ABLATION LEADERBOARD ===')
print(abl_df.to_string())
abl_df.to_csv('ablation_leaderboard.csv')
print('\nSaved ablation_leaderboard.csv')

T5 GEC:   0%|          | 0/4 [00:00<?, ?it/s]

T5 GEC:  25%|██▌       | 1/4 [00:02<00:06,  2.08s/it]

T5 GEC:  50%|█████     | 2/4 [00:05<00:06,  3.03s/it]

T5 GEC:  75%|███████▌  | 3/4 [00:09<00:03,  3.19s/it]

T5 GEC: 100%|██████████| 4/4 [00:10<00:00,  2.65s/it]

...

CoLA: 100%|██████████| 1/1 [00:01<00:00,  1.22s/it]

CoLA:   0%|          | 0/1 [00:00<?, ?it/s]

CoLA: 100%|██████████| 1/1 [00:01<00:00,  1.25s/it]

CoLA: 100%|██████████| 1/1 [00:01<00:00,  1.25s/it]

  f0.5 = 0.483

=== ABLATION LEADERBOARD ===
                 f0.5  cola_B  cola_C  sem_sim    B<A    C>B
config                                                      
zero_shot       0.598     0.0     0.0    0.097  False  False
instr_concise   0.551     0.0     0.0    0.071  False  False
instr_detailed  0.501     0.0     0.0    0.089  False  False
few_shot_12     0.499     0.0     0.0    0.052  False  False
temp_1.0        0.483     0.0     0.0    0.080  False  False
one_shot        0.475     0.0     0.0    0.097  False  False
few_shot_3      0.466     0.0     0.0    0.074  False  False
temp_0.3        0.455     0.0     0.0    0.087  False  False
cot             0.391     0.0     0.0    0.044  False  False

Saved ablation_leaderboard.csv
